Übung:  Gateway API (microk8s + traefik)
-------------------

Gateway API ist ein offizielles Kubernetes-Projekt, das sich auf L4- und L7-Routing in Kubernetes konzentriert. Dieses Projekt stellt die nächste Generation von Kubernetes Ingress-, Load Balancing- und Service Mesh-APIs dar. Es war von Anfang an generisch, ausdrucksstark und rollenorientiert konzipiert.

Das Gesamtressourcenmodell konzentriert sich auf drei separate [Personas](https://gateway-api.sigs.k8s.io/concepts/roles-and-personas) (Infrastructur, Cluster, Application) und entsprechende Ressourcen, die von ihnen verwaltet werden sollen:

![](https://gateway-api.sigs.k8s.io/images/resource-model.png)

Quelle: [Gateway API](https://gateway-api.sigs.k8s.io/)
- - -

Um die Gateway-API-Ressourcen verwenden zu können, müssen diese zunächst erstellt werden, da sie nicht Teil des Standard-Kubernetes sind.

Anschliessend muss ein Gateway-fähiger Ingress-Controller installiert werden. In diesem Setup wird Traefik verwendet, welcher Gateway API nativ unterstützt.

Weitere Informationen zur Erstellung der Ressourcen und unterstützten Gateway-Controllern findest du [hier](https://gateway-api.sigs.k8s.io/guides/?utm_source=chatgpt.com#installing-a-gateway-controller).

In [ ]:
%%bash
GATEWAY_API_VERSION="v1.5.1"
kubectl apply --server-side -f "https://github.com/kubernetes-sigs/gateway-api/releases/download/${GATEWAY_API_VERSION}/standard-install.yaml"

### Ingress Service (Traefik)

In der aktuellen Umgebung übernimmt Traefik die Gateway-API-Funktionalität. Traefik läuft als Pods im Namespace `kube-system`.

Es wird nur die Gateway API Variante aktiviert, d.h. Traefik unterstützt kein "altes" Ingress.

In [ ]:
%%bash
helm repo add traefik https://traefik.github.io/charts
helm repo update
helm upgrade --install traefik traefik/traefik \
  --namespace kube-system \
  --set providers.kubernetesIngress.enabled=false \
  --set ingressClass.enabled=false \
  --set providers.kubernetesGateway.enabled=true \
  --set providers.kubernetesCRD.enabled=true \
  --set gatewayClass.enabled=true \
  --set gatewayClass.name=traefik \
  --set gateway.enabled=false \
  --set service.type=NodePort \
  --set ports.web.nodePort=31080 \
  --set ports.websecure.nodePort=31443 \
  --set ports.traefik.expose.default=true \
  --set ports.traefik.nodePort=31900 \
  --set api.dashboard=true \
  --set ingressRoute.dashboard.enabled=true

Durch die Verwendung von GatewayClass können Cluster-Administratoren standardisierte Gateway-Konfigurationen (Ingress Controller, Reverse Proxy) bereitstellen, die von verschiedenen Teams oder Anwendungen innerhalb des Clusters genutzt werden können.

Beim Installieren des Cluster haben die Persona **"Infrastructure Provider"** die GatewayClass erstellt:

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1
kind: GatewayClass
metadata:
  name: traefik
spec:
  controllerName: traefik.io/gateway-controller
EOF

Ein Gateway fungiert als Einstiegspunkt für den eingehenden Datenverkehr und leitet Anfragen basierend auf den definierten Regeln und Konfigurationen weiter. 

Gateways erstellen die Persona **"Cluster Operators"**.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: traefik
  namespace: kube-system
spec:
  gatewayClassName: traefik
  listeners:
    - name: http
      protocol: HTTP
      port: 8000
      allowedRoutes:
        namespaces:
          from: All
EOF

Nach dem Einrichten der benötigten Ressourcen, für das Gateway API, kontrollieren wir ob `GatewayClass` und `Gateways` auf "True" stehen. Dann war die Konfiguration erfolgreich.

In [ ]:
%%bash
kubectl get gatewayclass traefik
kubectl -n kube-system get gateways traefik
# echo "----------------------------"
# kubectl -n kube-system describe gateway traefik

- - -

## Microservices

Das Beispiel besteht aus drei Microservices: **Order**, **Customer** und **Catalog**. 

**Order** nutzt **Catalog** und **Customer** mit der REST-Schnittstelle. Ausserdem bietet jeder Microservice einige HTML-Seiten an.

Statt des ersten Microservices **Webshop**, der als Reverse Proxy konfiguriert ist, wird das Kubernetes Gateway API verwendet.

- - -

Wir Starten die Microservices wie zuvor. Dabei ist zu Beachten, dass die Persona **"Application Developers"** für die Services zuständig sind.

In [ ]:
%%bash
kubectl create namespace ms-gateway
kubectl apply -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-pod/catalog-pod.yaml -n ms-gateway
kubectl apply -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-pod/customer-pod.yaml -n ms-gateway
kubectl apply -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/2.1.0-pod/order-pod.yaml -n ms-gateway
kubectl apply -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/catalog-service.yaml -n ms-gateway
kubectl apply -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/customer-service.yaml -n ms-gateway
kubectl apply -f https://gitlab.com/ch-mc-b/autoshop-ms/infra/kubernetes-templates/-/raw/main/order-service.yaml -n ms-gateway

Die HTTPRoute übernimmt das Routing von HTTP- und HTTPS-Anfragen innerhalb eines Clusters. 

Sie spezifiziert, wie Anfragen, die ein Gateway passieren, zu den entsprechenden Services weitergeleitet werden. 

Die HTTP Routen sind in der Verantwortung der Persona **"Application Developer"**.

**ACHTUNG**: läuft diese Notebook nicht in einer Hyper-V Umgebung, muss der Eintrag `hostname` angepasst werden.

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: ms-gateway-route
  namespace: ms-gateway
spec:
  parentRefs:
    - group: gateway.networking.k8s.io
      kind: Gateway
      name: traefik
      namespace: kube-system
  hostnames:
    - "$(cat ~/data/server-ip)"
  rules:
    - matches:
        - path:
            type: PathPrefix
            value: /order
      backendRefs:
        - name: order
          port: 8080
    - matches:
        - path:
            type: PathPrefix
            value: /customer
      backendRefs:
        - name: customer
          port: 8080
    - matches:
        - path:
            type: PathPrefix
            value: /catalog
      backendRefs:
        - name: catalog
          port: 8080
EOF

Überprüfen der erstellen Ressourcen

In [ ]:
! kubectl get pod,services,endpoints,httproute -n ms-gateway -o wide
! # echo "----------------------------"
! # kubectl describe httproute ms-gateway-route -n ms-gateway

Alle Microservices sind jetzt mittels gleichem DNS erreichbar.

In [ ]:
%%bash
export SERVER=http://$(cat ~/data/server-ip):31080
echo "Kunden      : ${SERVER}/customer"
echo "Produkte    : ${SERVER}/catalog"
echo "Bestellungen: ${SERVER}/order"

***
### Ingress Service (Traefik)

In der aktuellen Umgebung übernimmt Traefik die Ingress-Funktionalität. Traefik läuft als Pods im Namespace `traefik`.

Von Traefik können wir uns die aktuelle Konfiguration beziehungsweise den Status über die Kubernetes-Ressourcen ausgeben lassen:

In [ ]:
%%bash
echo "Traefik Dashboard:" http://$(cat ~/data/server-ip):31900/dashboard/
curl http://$(cat ~/data/server-ip):31900/api/http/routers | jq

Zum Testen kann der `kubectl apply -f -` welche die HTTPRoute Ressourcen anlegt, durch `kubectl delete -f -` ersetzt werden und dann der obige Befehl wieder ausgeführt werden.

Dann sollten die Einträge für `customer`, `catalog` und `order` nicht mehr vorhanden sein.

- - -

### Aufräumen

In [ ]:
! kubectl delete namespace ms-gateway